In [1]:
from pathlib import Path
import pandas as pd
import re

import warnings
warnings.filterwarnings('ignore')

import os
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             roc_auc_score, precision_recall_curve, ConfusionMatrixDisplay, balanced_accuracy_score,
                             silhouette_score)
from sklearn.cluster import KMeans
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import KernelPCA
import joblib

import joblib, os
import numpy as np
import pandas as pd
import torch

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
import shap
from scipy.stats import spearmanr

# Set reproducibility seeds
os.environ['PYTHONHASHSEED'] = '42'
np.random.seed(42)
torch.manual_seed(42)
import random
random.seed(42)

# Reproducible torch behavior where possible
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

In [2]:
# Load the raw tables
DATA_DIR = Path("/home/gandalf/Documents/Data/Deep Mut/")
MUT_INFO_PATH   = DATA_DIR / 'mut_info.tsv'
RNASEQ_PATH = DATA_DIR / "rnaseq_data.tsv"
META_PATH = DATA_DIR / "meta_data.tsv"

# Load metadata normally
meta_df = pd.read_csv(META_PATH, sep="\t")

# Read first row for gene names
header_df = pd.read_csv(RNASEQ_PATH, sep="\t", header=None, nrows=1, dtype=str)
gene_names = header_df.iloc[0, :].tolist()

# Read data rows (skip header)
rnaseq_df = pd.read_csv(RNASEQ_PATH, sep="\t", header=None, skiprows=1, dtype=str, engine='python', on_bad_lines='warn')

# Pad/truncate to match header length
if rnaseq_df.shape[1] > len(gene_names):
    rnaseq_df = rnaseq_df.iloc[:, :len(gene_names)]
elif rnaseq_df.shape[1] < len(gene_names):
    for i in range(len(gene_names) - rnaseq_df.shape[1]):
        rnaseq_df[len(rnaseq_df.columns)] = None

rnaseq_df.columns = gene_names

# Rename first column to "sample_id"
rnaseq_df = rnaseq_df.rename(columns={gene_names[0]: 'sample_id'})

print(f"RNA-seq shape: {rnaseq_df.shape}")

RNA-seq shape: (9626, 19311)


In [3]:
sorted(meta_df["histological_grade"].dropna().unique())

['G1',
 'G2',
 'G3',
 'G4',
 'GB',
 'GX',
 'High Grade',
 'Low Grade',
 '[Discrepancy]',
 '[Unknown]']

In [4]:
TCGA_ID_PATTERN = re.compile(r"TCGA-[A-Z0-9]+-[A-Z0-9]+-\d+")


def _clean_meta(df: pd.DataFrame) -> pd.DataFrame:
    meta = df.copy()
    meta.columns = (
        meta.columns.astype(str)
        .str.strip()
        .str.strip('"')
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    
    if 'sample' in meta.columns:
        meta['patient_id'] = meta['sample'].astype(str).str.strip().str.replace('"', '')

    
    meta = meta.dropna(subset=['patient_id']).drop_duplicates('patient_id')

    if "histological_grade" in meta.columns:
        meta["histological_grade"] = (
            meta["histological_grade"].astype(str).str.strip().str.upper().replace({"": pd.NA})
        )

    return meta


def _clean_rnaseq(df: pd.DataFrame) -> pd.DataFrame:
    """
    RNA-seq data structure:
    - Header row: gene names (19311 fields)
    - Data rows: TCGA sample ID in col 1, then 19310 expression values
    """
    # Extract gene names from header
    gene_names = df.columns.astype(str).str.strip().str.replace('"', '')
    
    # Extract sample IDs from first column (no normalization)
    sample_ids = df.iloc[:, 0].astype(str).str.strip().str.replace('"', '')
    
    # Extract expression data: columns 2 onwards
    expr_data = df.iloc[:, 1:].copy()
    expr_data.columns = gene_names[:len(expr_data.columns)]
    expr_data = expr_data.apply(pd.to_numeric, errors="coerce")
    
    # Create dataframe with sample_id and gene expressions
    rna_clean = pd.concat([
        sample_ids.reset_index(drop=True),
        expr_data.reset_index(drop=True)
    ], axis=1)
    
    rna_clean.columns = ['patient_id'] + list(expr_data.columns)
    
    # Remove any empty sample IDs
    rna_clean = rna_clean[rna_clean['patient_id'].str.len() > 0].dropna(subset=['patient_id'])
    
    rna_clean = rna_clean.set_index('patient_id')
    
    # Handle duplicate patients by averaging
    rna_final = rna_clean.groupby(level=0).mean()
    rna_final = rna_final.T  # genes x patients
    rna_final['gene_id'] = rna_final.index
    rna_final = rna_final[['gene_id'] + [c for c in rna_final.columns if c != 'gene_id']]
    rna_final = rna_final.reset_index(drop=True)
    
    return rna_final


clean_meta_df = _clean_meta(meta_df)
clean_rnaseq_df = _clean_rnaseq(rnaseq_df)

rnaseq_patient_ids = set(clean_rnaseq_df.columns) - {"gene_id"}
matched_patient_ids = sorted(set(clean_meta_df["patient_id"]) & rnaseq_patient_ids)

clean_meta_df = clean_meta_df[clean_meta_df["patient_id"].isin(matched_patient_ids)].reset_index(drop=True)
clean_rnaseq_df = clean_rnaseq_df[["gene_id"] + matched_patient_ids]

print(f"Matched {len(matched_patient_ids)} patients across both tables")
print(f"Metadata: {len(clean_meta_df)} rows")
print(f"RNA-seq: {len(clean_rnaseq_df)} genes x {len(matched_patient_ids)} samples")
clean_meta_df.head()

Matched 9626 patients across both tables
Metadata: 9626 rows
RNA-seq: 19310 genes x 9626 samples


,sample_type_id,sample_type,project_id,rnaseqid,mutid,sample,x_patient,cancer.type.abbreviation,age_at_initial_pathologic_diagnosis,gender,...,os.time,dss,dss.time,dfi,dfi.time,pfi,pfi.time,redaction,x_primary_disease,patient_id
0,1,Primary Tumor,TCGA-GBM,TCGA-02-0047-01A,TCGA-02-0047-01,TCGA-02-0047-01,TCGA-02-0047,GBM,78.0,MALE,...,448.0,1.0,448.0,NaN,NaN,1.0,57.0,NaN,glioblastoma multiforme,TCGA-02-0047-01
1,1,Primary Tumor,TCGA-GBM,TCGA-02-0055-01A,TCGA-02-0055-01,TCGA-02-0055-01,TCGA-02-0055,GBM,62.0,FEMALE,...,76.0,1.0,76.0,NaN,NaN,1.0,6.0,NaN,glioblastoma multiforme,TCGA-02-0055-01
2,1,Primary Tumor,TCGA-GBM,TCGA-02-2483-01A,TCGA-02-2483-01,TCGA-02-2483-01,TCGA-02-2483,GBM,43.0,MALE,...,466.0,0.0,466.0,NaN,NaN,0.0,466.0,NaN,glioblastoma multiforme,TCGA-02-2483-01
3,1,Primary Tumor,TCGA-GBM,TCGA-02-2485-01A,TCGA-02-2485-01,TCGA-02-2485-01,TCGA-02-2485,GBM,53.0,MALE,...,470.0,0.0,470.0,NaN,NaN,1.0,186.0,NaN,glioblastoma multiforme,TCGA-02-2485-01
4,1,Primary Tumor,TCGA-GBM,TCGA-02-2486-01A,TCGA-02-2486-01,TCGA-02-2486-01,TCGA-02-2486,GBM,64.0,MALE,...,618.0,1.0,618.0,NaN,NaN,1.0,618.0,NaN,glioblastoma multiforme,TCGA-02-2486-01


In [5]:
def save_gene_order_json(rnaseq_df, output_path):
    gene_order = (
        rnaseq_df["gene_id"]
        .astype(str)
        .str.strip()
        .str.strip('"')
        .tolist()
    )

    if len(gene_order) != len(set(gene_order)):
        raise ValueError("Duplicate gene names found in gene_order.")

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w") as f:
        json.dump(gene_order, f, indent=2)

    print(f"Saved {len(gene_order)} genes to {output_path}")

save_gene_order_json(clean_rnaseq_df, "deployment_grade/gene_order.json")

Saved 19310 genes to deployment_grade/gene_order.json


In [5]:
meta_test = meta_df.copy()
cols_lower = (
    meta_test.columns.astype(str)
    .str.strip()
    .str.strip('"')
    .str.lower()
    .str.replace(" ", "_", regex=False)
)
meta_test.columns = cols_lower

id_candidates = ['sample', 'x_patient', 'rnaseqid', 'sample_type_id', 'mutid']

def _normalize_patient_id(val):
    if pd.isna(val):
        return pd.NA
    s = str(val).strip().replace('"', '')
    return s if s else pd.NA

def _select_patient_id_test(row):
    for col in id_candidates:
        val = row.get(col)
        normalized = _normalize_patient_id(val)
        print(f"    {col}: {val} -> {normalized}")
        if not pd.isna(normalized):
            return normalized
    return pd.NA

In [6]:
print("Meta patient_id):")
print(clean_meta_df['patient_id'].head().tolist())

print("\nRNA-seq patient_id:")
print(list(rnaseq_patient_ids)[:5])

print(f"\nMeta unique count: {len(set(clean_meta_df['patient_id']))}")
print(f"RNA-seq unique count: {len(rnaseq_patient_ids)}")

print("\nIntersection:")
intersection = set(clean_meta_df["patient_id"]) & rnaseq_patient_ids
print(f"Matched: {len(intersection)}")

Meta patient_id):
['TCGA-02-0047-01', 'TCGA-02-0055-01', 'TCGA-02-2483-01', 'TCGA-02-2485-01', 'TCGA-02-2486-01']

RNA-seq patient_id:
['TCGA-60-2723-01', 'TCGA-RY-A845-01', 'TCGA-TS-A8AV-01', 'TCGA-IB-7645-01', 'TCGA-EJ-5510-01']

Meta unique count: 9626
RNA-seq unique count: 9626

Intersection:
Matched: 9626


In [7]:
print("=== DATASET BEFORE LABELING ===")
print(f"Metadata shape: {clean_meta_df.shape}")
print(f"RNA-seq shape: {clean_rnaseq_df.shape}")

# Map cancer grades to binary risk labels
# low risk: G1 / LOW GRADE
# high risk: G3 / G4 / HIGH GRADE
low_risk_grades = {'G1', 'LOW GRADE'}
high_risk_grades = {'G3', 'G4', 'HIGH GRADE'}

def grade_to_risk_label(grade):
    """Map histological grade to binary risk label."""
    if pd.isna(grade):
        return pd.NA
    grade_str = str(grade).strip().upper()
    if grade_str in low_risk_grades:
        return 0  # Low risk
    elif grade_str in high_risk_grades:
        return 1  # High risk
    else:
        return pd.NA  # Unknown

clean_meta_df['risk_label'] = clean_meta_df['histological_grade'].apply(grade_to_risk_label)

# Filter to only samples with valid risk labels
valid_idx = clean_meta_df['risk_label'].notna()
labeled_meta_df = clean_meta_df[valid_idx].reset_index(drop=True)

# Filter RNA-seq to match
valid_patient_ids = set(labeled_meta_df['patient_id'])
labeled_rnaseq_df = clean_rnaseq_df[['gene_id'] + sorted([p for p in clean_rnaseq_df.columns if p in valid_patient_ids])].reset_index(drop=True)

print("\n=== DATASET AFTER LABELING ===")
print(f"Metadata shape: {labeled_meta_df.shape}")
print(f"RNA-seq shape: {labeled_rnaseq_df.shape}")
print(f"\nRisk label distribution:")
print(labeled_meta_df['risk_label'].value_counts().sort_index())
print(f"  0 (Low risk):  {(labeled_meta_df['risk_label'] == 0).sum()}")
print(f"  1 (High risk): {(labeled_meta_df['risk_label'] == 1).sum()}")

=== DATASET BEFORE LABELING ===
Metadata shape: (9626, 41)
RNA-seq shape: (19310, 9627)

=== DATASET AFTER LABELING ===
Metadata shape: (2494, 42)
RNA-seq shape: (19310, 2495)

Risk label distribution:
risk_label
0     312
1    2182
Name: count, dtype: int64
  0 (Low risk):  312
  1 (High risk): 2182


### Using KernelPCA + Neural Feature Selector + XGBoost

#### =============================================================================
#### STAGE 2: KMeans Silhouette Ranking on Components
#### =============================================================================

In [8]:
def stage_0_prepare_data(X, y, test_size=0.15, val_size=0.15, random_state=42, output_dir='deployment_grade'):

    os.makedirs(output_dir, exist_ok=True)
    
    # Stratified split: train+val vs test
    splitter1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_val_idx, test_idx = next(splitter1.split(X, y))
    
    X_train_val = X[train_val_idx]
    y_train_val = y[train_val_idx]
    X_test = X[test_idx]
    y_test = y[test_idx]
    
    # Stratified split: train vs val
    val_size_adjusted = val_size / (1.0 - test_size)
    splitter2 = StratifiedShuffleSplit(n_splits=1, test_size=val_size_adjusted, random_state=random_state)
    train_idx, val_idx = next(splitter2.split(X_train_val, y_train_val))
    
    X_train = X_train_val[train_idx]
    y_train = y_train_val[train_idx]
    X_val = X_train_val[val_idx]
    y_val = y_train_val[val_idx]
    
    # Fit scaler on train, apply to all
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # Save scaler
    scaler_path = os.path.join(output_dir, 'scaler.pkl')
    joblib.dump(scaler, scaler_path)
    
    result = {
        'X_train': X_train_scaled,
        'y_train': y_train,
        'X_val': X_val_scaled,
        'y_val': y_val,
        'X_test': X_test_scaled,
        'y_test': y_test,
        'scaler': scaler,
        'scaler_path': scaler_path,
        'n_features': X.shape[1],
    }
    
    print(f"STAGE 0: Data Prepared")
    print(f"  Train: {result['X_train'].shape} | Val: {result['X_val'].shape} | Test: {result['X_test'].shape}")
    print(f"  Label distribution:")
    print(f"    Train - Low: {(y_train==0).sum()}, High: {(y_train==1).sum()}")
    print(f"    Val   - Low: {(y_val==0).sum()}, High: {(y_val==1).sum()}")
    print(f"    Test  - Low: {(y_test==0).sum()}, High: {(y_test==1).sum()}")
    
    return result

X_raw = labeled_rnaseq_df.drop('gene_id', axis=1).T.values  # (samples x genes)
y_binary = labeled_meta_df['risk_label'].values

data = stage_0_prepare_data(X_raw, y_binary, output_dir='deployment_grade')


STAGE 0: Data Prepared
  Train: (1745, 19310) | Val: (374, 19310) | Test: (375, 19310)
  Label distribution:
    Train - Low: 218, High: 1527
    Val   - Low: 47, High: 327
    Test  - Low: 47, High: 328


#### =============================================================================
#### STAGE 1: Unsupervised Components (KernelPCA)
#### =============================================================================

In [11]:
def stage_1_unsupervised_components(X_train, X_val, method='kpca', n_components=100,
                                   output_dir='deployment_grade', kernel='rbf'):
    method = str(method).lower().strip()
    os.makedirs(output_dir, exist_ok=True)

    print(f"  [{method.upper()}] X_train shape: {X_train.shape}")

    t0 = time.time()
    if method == 'kpca':
        component_op = Pipeline([
            ('scaler', StandardScaler()),
            ('kpca', KernelPCA(n_components=n_components, kernel=kernel, fit_inverse_transform=False))
        ])
        X_train_components = component_op.fit_transform(X_train)
        X_val_components = component_op.transform(X_val)
        component_name = f"KPCA_{kernel.upper()}"

    else:
        raise ValueError(f'Unknown method: {method}. Use kpca')

    train_time = time.time() - t0
    infer_per_sample_ms = (train_time / max(len(X_train) + len(X_val), 1)) * 1000

    model_path = os.path.join(output_dir, f'{method}.pkl')
    joblib.dump(component_op, model_path)

    config = {
        'method': method,
        'n_components': int(n_components),
        'random_state': 42,
    }
    if method == 'kpca':
        config.update({'kernel': str(kernel)})

    config_path = os.path.join(output_dir, f'{method}_config.json')
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)

    print(f"STAGE 1: {component_name} fitted")
    print(f"  X_train_components: {X_train_components.shape}")
    print(f"  X_val_components: {X_val_components.shape}")
    print(f"  Train time: {train_time:.2f}s")
    print(f"  Infer time/sample (val): {infer_per_sample_ms:.3f}ms")

    return {
        'method': method,
        'component_name': component_name,
        'component_op': component_op,
        'X_train_components': X_train_components,
        'X_val_components': X_val_components,
        'model_path': model_path,
        'config_path': config_path,
        'train_time_s': train_time,
        'infer_time_per_sample_ms': infer_per_sample_ms,
    }


def transform_components(component_result, X):
    return component_result['component_op'].transform(X)


RUN_ALL_COMPONENT_METHODS = True
ACTIVE_COMPONENT_METHOD = 'kpca_rbf'

COMPONENT_METHOD_CONFIGS = [
    {'name': 'kpca_rbf', 'method': 'kpca', 'n_components': 100, 'kernel': 'rbf'},
    {'name': 'kpca_cosine', 'method': 'kpca', 'n_components': 100, 'kernel': 'cosine'},
]

component_results = {}
for cfg in COMPONENT_METHOD_CONFIGS:
    method_name = cfg['name']
    try:
        component_results[method_name] = stage_1_unsupervised_components(
            data['X_train'],
            data['X_val'],
            method=cfg.get('method', method_name),
            n_components=cfg.get('n_components', 100),
            output_dir=f"deployment_grade/{method_name}",
            kernel=cfg.get('kernel', 'rbf'),
        )
    except Exception as e:
        print(f"  Skipping {method_name.upper()} Stage 1: {e}")

if ACTIVE_COMPONENT_METHOD not in component_results and len(component_results) > 0:
    ACTIVE_COMPONENT_METHOD = list(component_results.keys())[0]
    print(f"  ACTIVE_COMPONENT_METHOD fallback -> {ACTIVE_COMPONENT_METHOD}")

component_result = component_results[ACTIVE_COMPONENT_METHOD]
print(f"\nUsing {component_result['component_name']} components for downstream stages")

  [KPCA] X_train shape: (1745, 19310)
STAGE 1: KPCA_RBF fitted
  X_train_components: (1745, 100)
  X_val_components: (374, 100)
  Train time: 1.05s
  Infer time/sample (val): 0.498ms
  [KPCA] X_train shape: (1745, 19310)
STAGE 1: KPCA_COSINE fitted
  X_train_components: (1745, 100)
  X_val_components: (374, 100)
  Train time: 0.64s
  Infer time/sample (val): 0.303ms

Using KPCA_RBF components for downstream stages


#### =============================================================================
#### STAGE 2: KMeans Silhouette Component Ranking
#### =============================================================================

In [12]:
def stage_2_kmeans_silhouette_ranking(X_train_components, component_name='Components',
                                      top_k=50, output_dir='deployment_grade', n_clusters=4):
    """
    Unsupervised ranking by single-component KMeans cluster-separation score.
    For each component i, run 1D KMeans and score with absolute silhouette.
    """
    print(f"  Ranking {X_train_components.shape[1]} {component_name} components using KMeans silhouette...")

    component_scores = np.zeros(X_train_components.shape[1])

    for i in range(X_train_components.shape[1]):
        xi = X_train_components[:, [i]]
        try:
            kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
            labels = kmeans.fit_predict(xi)
            if len(np.unique(labels)) < 2:
                component_scores[i] = 0.0
            else:
                score = silhouette_score(xi, labels, sample_size=None)
                component_scores[i] = abs(float(score))
        except Exception:
            component_scores[i] = 0.0

    ranked_indices = np.argsort(component_scores)[::-1]
    top_k_indices = ranked_indices[:top_k].tolist()

    print(f"  Top-{top_k} components (by silhouette score, {component_name}):")
    for i, idx in enumerate(top_k_indices[:5]):
        print(f"    {i+1}. Component {idx}: {component_scores[idx]:.6f}")

    os.makedirs(output_dir, exist_ok=True)
    safe_name = component_name.lower().replace('-', '').replace(' ', '')
    topk_path = os.path.join(output_dir, f'top_k_components_{safe_name}_kmeans_silhouette.json')
    with open(topk_path, 'w') as f:
        json.dump({
            'top_k': int(top_k),
            'component_indices': top_k_indices,
            'component_scores': component_scores.tolist(),
            'ranking_method': 'silhouette_cluster_separation',
            'clustering_method': 'kmeans',
            'n_clusters': int(n_clusters),
            'component_method': component_name,
        }, f, indent=2)

    fig, ax = plt.subplots(figsize=(10, 6))
    top_indices_for_plot = ranked_indices[:20]
    scores_for_plot = component_scores[top_indices_for_plot]
    ax.barh(range(len(top_indices_for_plot)), scores_for_plot)
    ax.set_yticks(range(len(top_indices_for_plot)))
    ax.set_yticklabels([f'Component {i}' for i in top_indices_for_plot])
    ax.set_xlabel('Silhouette Score (abs)')
    ax.set_title(f'KMeans Silhouette Ranking: Top 20 {component_name} Components')
    ax.invert_yaxis()
    plt.tight_layout()
    ranking_plot_path = os.path.join(output_dir, f'kmeans_silhouette_ranking_bar_{safe_name}.png')
    plt.savefig(ranking_plot_path, dpi=150, bbox_inches='tight')
    plt.close()

    print(f" STAGE 2 ({component_name}): KMeans silhouette ranking complete")
    print(f"  Top-K components saved to: {topk_path}")

    return {
        'top_k_indices': top_k_indices,
        'component_scores': component_scores,
        'topk_path': topk_path,
        'ranking_plot_path': ranking_plot_path,
        'clustering_method': 'kmeans',
    }


shap_results = {}

for method_name, cr in component_results.items():
    print(f"\n=== Stage 2 (KMeans silhouette) for {cr['component_name']} ({method_name}) ===")
    shap_results[method_name] = stage_2_kmeans_silhouette_ranking(
        cr['X_train_components'],
        component_name=cr['component_name'],
        top_k=50,
        output_dir=f"deployment_grade/{method_name}",
        n_clusters=4,
    )

shap_result = shap_results[ACTIVE_COMPONENT_METHOD]


=== Stage 2 (KMeans silhouette) for KPCA_RBF (kpca_rbf) ===
  Ranking 100 KPCA_RBF components using KMeans silhouette...
  Top-50 components (by silhouette score, KPCA_RBF):
    1. Component 1: 0.586171
    2. Component 0: 0.576294
    3. Component 5: 0.573006
    4. Component 7: 0.563127
    5. Component 3: 0.562900
 STAGE 2 (KPCA_RBF): KMeans silhouette ranking complete
  Top-K components saved to: deployment_grade/kpca_rbf/top_k_components_kpca_rbf_kmeans_silhouette.json

=== Stage 2 (KMeans silhouette) for KPCA_COSINE (kpca_cosine) ===
  Ranking 100 KPCA_COSINE components using KMeans silhouette...
  Top-50 components (by silhouette score, KPCA_COSINE):
    1. Component 3: 0.577676
    2. Component 0: 0.574924
    3. Component 1: 0.554570
    4. Component 6: 0.544889
    5. Component 2: 0.543242
 STAGE 2 (KPCA_COSINE): KMeans silhouette ranking complete
  Top-K components saved to: deployment_grade/kpca_cosine/top_k_components_kpca_cosine_kmeans_silhouette.json


#### =============================================================================
#### STAGE 3: Train MLP Feature Selector
#### =============================================================================

In [13]:
class FeatureAttentionBlock(nn.Module):
    def __init__(self, dim, dropout=0.2, se_ratio=0.25):
        super().__init__()
        hidden = max(32, dim // 2)
        se_hidden = max(16, int(dim * se_ratio))

        self.norm = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, dim),
            nn.Dropout(dropout),
        )
        self.gate = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Sigmoid(),
        )
        self.channel_attn = nn.Sequential(
            nn.Linear(dim, se_hidden),
            nn.ReLU(),
            nn.Linear(se_hidden, dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        h = self.norm(x)
        ff_out = self.ff(h)
        gated = ff_out * self.gate(h)
        x = x + gated
        scale = self.channel_attn(x)
        x = x * scale + x
        return x


class AttentionFeatureSelectorMLP(nn.Module):
    def __init__(self, n_input_genes, n_output_components, hidden_dims=None, dropout=0.4):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [1024, 512, 256, 128]

        self.stem = nn.Sequential(
            nn.Linear(n_input_genes, hidden_dims[0]),
            nn.BatchNorm1d(hidden_dims[0]),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        blocks = []
        transitions = []
        for i, dim in enumerate(hidden_dims):
            blocks.append(FeatureAttentionBlock(dim, dropout=dropout))
            if i < len(hidden_dims) - 1:
                transitions.append(
                    nn.Sequential(
                        nn.Linear(dim, hidden_dims[i + 1]),
                        nn.BatchNorm1d(hidden_dims[i + 1]),
                        nn.GELU(),
                        nn.Dropout(dropout),
                    )
                )
        self.blocks = nn.ModuleList(blocks)
        self.transitions = nn.ModuleList(transitions)

        self.component_head = nn.Linear(hidden_dims[-1], n_output_components)

    def forward(self, x):
        h = self.stem(x)
        for i, block in enumerate(self.blocks):
            h = block(h)
            if i < len(self.transitions):
                h = self.transitions[i](h)
        return self.component_head(h)


def compute_distance_matrix(X_components):
    """Pairwise Euclidean distances in component space."""
    from sklearn.metrics import pairwise_distances
    return pairwise_distances(X_components, metric='euclidean')


def mine_triplets_from_distances(dist_matrix, n_triplets=512, random_state=42):
    """
    For each anchor, select the closest sample as positive and a
    random distant sample (top-50% distance) as negative.
    Returns (anchor_idx, positive_idx, negative_idx) arrays.
    No labels used.
    """
    rng = np.random.RandomState(random_state)
    n = dist_matrix.shape[0]
    anchors, positives, negatives = [], [], []

    for _ in range(n_triplets):
        anchor = rng.randint(0, n)
        dists = dist_matrix[anchor].copy()
        dists[anchor] = np.inf

        positive = int(np.argmin(dists))

        median_dist = np.median(dists[dists < np.inf])
        far_candidates = np.where(dists > median_dist)[0]
        if len(far_candidates) == 0:
            far_candidates = np.argsort(dists)[-10:]
        negative = int(rng.choice(far_candidates))

        anchors.append(anchor)
        positives.append(positive)
        negatives.append(negative)

    return np.array(anchors), np.array(positives), np.array(negatives)


def stage_3_train_component_predictor(X_train, X_val, X_train_components, X_val_components,
                                      top_k_indices, output_dir='deployment_grade',
                                      epochs=100, batch_size=32, lr=1e-3, patience=10,
                                      y_train_labels=None, y_val_labels=None,
                                      component_loss_weight=1.0, grade_loss_weight=1.0,
                                      random_state=42):
    """
    Attention-based reconstruction model (no joint loss).
    Learns to predict selected top-K components from gene expression.
    Computes training loss in eval mode on full training set for direct comparison with validation loss.
    """

    os.makedirs(output_dir, exist_ok=True)

    # Reset RNG state per run so repeated executions with same settings are reproducible.
    random.seed(random_state)
    np.random.seed(random_state)
    torch.manual_seed(random_state)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(random_state)

    component_targets_train = X_train_components[:, top_k_indices]
    component_targets_val = X_val_components[:, top_k_indices]

    from sklearn.preprocessing import StandardScaler as TargetScaler

    target_scaler = TargetScaler()
    component_targets_train = target_scaler.fit_transform(component_targets_train)
    component_targets_val = target_scaler.transform(component_targets_val)

    print(f"  Component targets shape (train): {component_targets_train.shape}")
    print(f"  Component targets shape (val):   {component_targets_val.shape}")
    if grade_loss_weight != 0:
        print(f"  Note: grade_loss_weight={grade_loss_weight} is ignored (joint loss removed).")

    X_train_tensor = torch.FloatTensor(X_train)
    y_train_comp_tensor = torch.FloatTensor(component_targets_train)
    train_dataset = TensorDataset(X_train_tensor, y_train_comp_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Compute distance matrix and mine triplets (unsupervised)
    TRIPLET_LOSS_WEIGHT = 0.3
    dist_matrix = compute_distance_matrix(X_train_components)
    anc_idx, pos_idx, neg_idx = mine_triplets_from_distances(
        dist_matrix, n_triplets=1024, random_state=random_state
    )
    triplet_criterion = nn.TripletMarginLoss(margin=1.0, p=2)

    X_val_tensor = torch.FloatTensor(X_val)
    y_val_comp_tensor = torch.FloatTensor(component_targets_val)

    hidden_dims = [1024, 512, 256, 128]
    model = AttentionFeatureSelectorMLP(
        n_input_genes=X_train.shape[1],
        n_output_components=len(top_k_indices),
        hidden_dims=hidden_dims,
        dropout=0.2,
    )
    model.train()

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4)
    component_criterion = nn.MSELoss()

    online_train_losses = []  # batch-averaged training loss (during training)
    train_eval_losses = []     # model.eval() training loss (for comparison)
    val_component_losses = []

    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None

    print(f"  Training attention-based model for {epochs} epochs...")
    t_train = time.time()
    for epoch in range(epochs):
        # ===== Online training loss (batch-averaged) =====
        epoch_online_loss = 0.0

        model.train()
        for batch_X, batch_comp in train_loader:
            optimizer.zero_grad()
            pred_comp = model(batch_X)
            component_loss = component_criterion(pred_comp, batch_comp) * component_loss_weight
            component_loss.backward()

            optimizer.step()
            epoch_online_loss += component_loss.item()

        epoch_online_loss /= len(train_loader)
        online_train_losses.append(epoch_online_loss)

        # Triplet loss on predicted components (unsupervised geometry preservation)
        if len(anc_idx) > 0:
            model.train()
            optimizer.zero_grad()
            anc_t = torch.FloatTensor(X_train[anc_idx])
            pos_t = torch.FloatTensor(X_train[pos_idx])
            neg_t = torch.FloatTensor(X_train[neg_idx])
            pred_anc = model(anc_t)
            pred_pos = model(pos_t)
            pred_neg = model(neg_t)
            triplet_loss = triplet_criterion(pred_anc, pred_pos, pred_neg) * TRIPLET_LOSS_WEIGHT
            triplet_loss.backward()
            optimizer.step()

        # ===== Evaluation-mode training loss (full training set) =====
        model.eval()
        with torch.no_grad():
            train_pred_eval = model(X_train_tensor)
            train_loss_eval = component_criterion(train_pred_eval, y_train_comp_tensor)
            train_eval_loss = float(train_loss_eval.item()) * component_loss_weight
        train_eval_losses.append(train_eval_loss)

        # ===== Validation loss =====
        with torch.no_grad():
            val_pred = model(X_val_tensor)
            val_loss = component_criterion(val_pred, y_val_comp_tensor) * component_loss_weight
            val_loss_item = float(val_loss.item())
        val_component_losses.append(val_loss_item)

        # Early stopping and scheduler
        if val_loss_item < best_val_loss:
            best_val_loss = val_loss_item
            patience_counter = 0
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1

        scheduler.step(val_loss_item)

        if (epoch + 1) % max(1, epochs // 10) == 0:
            print(f"  Epoch {epoch + 1:3d}/{epochs}: train_eval_loss={train_eval_loss:.4f}, val_loss={val_loss_item:.4f}, patience={patience_counter}/{patience}")

        if patience_counter >= patience:
            print(f"  Early stopping at epoch {epoch + 1} (best val loss={best_val_loss:.4f})")
            break

    train_time = time.time() - t_train

    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    # Plot training curves
    fig, ax = plt.subplots(figsize=(10, 6))
    epochs_range = range(1, len(train_eval_losses) + 1)
    ax.plot(epochs_range, train_eval_losses, 'b-', linewidth=2, label='Train (eval mode, full set)', alpha=0.7)
    ax.plot(epochs_range, val_component_losses, 'r-', linewidth=2, label='Validation', alpha=0.7)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Component Loss (MSE)', fontsize=12)
    ax.set_title('Training Progress: Train (eval mode) vs Validation Loss', fontsize=13)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    loss_plot_path = os.path.join(output_dir, 'training_curves_eval_mode.png')
    plt.savefig(loss_plot_path, dpi=150, bbox_inches='tight')
    plt.close()

    # Save model
    model_path = os.path.join(output_dir, 'component_predictor_attention_mlp.pt')
    torch.save(model.state_dict(), model_path)

    result = {
        'model': model,
        'optimizer': optimizer,
        'best_val_loss': best_val_loss,
        'train_time_s': train_time,
        'online_train_losses': online_train_losses,
        'train_losses': train_eval_losses,
        'val_losses': val_component_losses,
        'train_component_losses': train_eval_losses,
        'val_component_losses': val_component_losses,
        'train_grade_losses': [],
        'val_grade_losses': [],
        'best_val_mse': best_val_loss,
        'train_time_s': train_time,
        'target_scaler': target_scaler,
    }

    print(f" STAGE 3: Attention Reconstruction Model Trained")
    print(f"  Best val component loss: {best_val_loss:.4f}")
    print(f"  Train time: {train_time:.2f}s")

    return result


In [15]:

MANUAL_COMPONENT_LOSS_WEIGHT = 1.0
MANUAL_GRADE_LOSS_WEIGHT = 0.0

MANUAL_STAGE3_RANDOM_STATE = 42
print(f'Manual Stage 3 seed: {MANUAL_STAGE3_RANDOM_STATE}')

mlp_results = {}
for method_name, cr in component_results.items():
    print(f"\n=== Stage 3 (KMeans ranking) for {cr['component_name']} ({method_name}) ===")
    method_output_dir = os.path.join('deployment_grade', method_name)

    mlp_results[method_name] = stage_3_train_component_predictor(
        data['X_train'], data['X_val'],
        cr['X_train_components'], cr['X_val_components'],
        shap_results[method_name]['top_k_indices'],
        output_dir=method_output_dir,
        epochs=200, batch_size=32, lr=1e-3, patience=20,
        y_train_labels=data['y_train'],
        y_val_labels=data['y_val'],
        component_loss_weight=MANUAL_COMPONENT_LOSS_WEIGHT,
        grade_loss_weight=MANUAL_GRADE_LOSS_WEIGHT,
        random_state=MANUAL_STAGE3_RANDOM_STATE,
    )

    mlp_results[method_name]['selected_weights'] = {
        'component_loss_weight': MANUAL_COMPONENT_LOSS_WEIGHT,
        'grade_loss_weight': MANUAL_GRADE_LOSS_WEIGHT,
    }

    print(
        f"  Using manual reconstruction weights for {method_name} (KMeans): "
        f"component={MANUAL_COMPONENT_LOSS_WEIGHT}"
    )

# Backward-compatible aliases for active method
mlp_result = mlp_results[ACTIVE_COMPONENT_METHOD]


Manual Stage 3 seed: 42

=== Stage 3 (KMeans ranking) for KPCA_RBF (kpca_rbf) ===
  Component targets shape (train): (1745, 50)
  Component targets shape (val):   (374, 50)
  Training attention-based model for 200 epochs...
  Epoch  20/200: train_eval_loss=0.2129, val_loss=0.1484, patience=1/20
  Epoch  40/200: train_eval_loss=0.1101, val_loss=0.0867, patience=1/20
  Epoch  60/200: train_eval_loss=0.0732, val_loss=0.0632, patience=2/20
  Epoch  80/200: train_eval_loss=0.0606, val_loss=0.0572, patience=8/20
  Epoch 100/200: train_eval_loss=0.0587, val_loss=0.0561, patience=11/20
  Early stopping at epoch 109 (best val loss=0.0556)
 STAGE 3: Attention Reconstruction Model Trained
  Best val component loss: 0.0556
  Train time: 498.93s
  Using manual reconstruction weights for kpca_rbf (KMeans): component=1.0

=== Stage 3 (KMeans ranking) for KPCA_COSINE (kpca_cosine) ===
  Component targets shape (train): (1745, 50)
  Component targets shape (val):   (374, 50)
  Training attention-based 

#### =============================================================================
#### STAGE 3b: Validate Selector
##### =============================================================================

In [16]:
def stage_3b_validate_selector(X_test, y_test, model,
                               top_k_indices, component_result,
                               target_scaler=None):
    """
    Validate MLP by comparing predicted top-K components against true selected-method components.
    """

    print("Applying MLP to test set...")

    model.eval()
    X_test_tensor = torch.FloatTensor(X_test)
    with torch.no_grad():
        pred_components_test = model(X_test_tensor).numpy()

    X_test_components = transform_components(component_result, X_test)
    true_components_test = X_test_components[:, top_k_indices]

    # Scale true components to match MLP output space
    if target_scaler is not None:
        true_components_test = target_scaler.transform(true_components_test)

    mse = float(np.mean((pred_components_test - true_components_test) ** 2))

    mean_true = np.mean(np.abs(true_components_test), axis=0)
    mean_pred = np.mean(np.abs(pred_components_test), axis=0)
    true_rank = np.argsort(-mean_true)
    pred_rank = np.argsort(-mean_pred)
    corrcoef, pval = spearmanr(true_rank, pred_rank)

    print(f" STAGE 3b: Component Predictor Validation ({component_result['component_name']})")
    print(f"  Test component MSE: {mse:.4f}")
    print(f"  Spearman r (true rank vs predicted rank): {corrcoef:.4f} (p={pval:.4e})")

    return {
        'component_mse': mse,
        'spearman_r': corrcoef,
        'spearman_p': pval,
        'pred_test_components': pred_components_test,
        'true_test_components': true_components_test,
    }


validation_results = {}
for method_name, cr in component_results.items():
    print(f"\n=== Stage 3b (KMeans ranking) for {cr['component_name']} ({method_name}) ===")
    validation_results[method_name] = stage_3b_validate_selector(
        data['X_test'], data['y_test'],
        mlp_results[method_name]['model'],
        shap_results[method_name]['top_k_indices'],
        cr,
        target_scaler=mlp_results[method_name]['target_scaler'],
    )

# Backward-compatible aliases for active method
validation_result = validation_results[ACTIVE_COMPONENT_METHOD]



=== Stage 3b (KMeans ranking) for KPCA_RBF (kpca_rbf) ===
Applying MLP to test set...
 STAGE 3b: Component Predictor Validation (KPCA_RBF)
  Test component MSE: 0.0579
  Spearman r (true rank vs predicted rank): -0.1116 (p=4.4018e-01)

=== Stage 3b (KMeans ranking) for KPCA_COSINE (kpca_cosine) ===
Applying MLP to test set...
 STAGE 3b: Component Predictor Validation (KPCA_COSINE)
  Test component MSE: 0.0956
  Spearman r (true rank vs predicted rank): 0.3597 (p=1.0296e-02)


#### =============================================================================
#### STAGE 4: Final XGBoost Predictor
#### =============================================================================

In [17]:
# Transform test set through both component methods before Stage 4
X_test_components_by_method = {}
for method_name, cr in component_results.items():
    X_test_components_by_method[method_name] = transform_components(cr, data['X_test'])
    print(f" {cr['component_name']} test shape: {X_test_components_by_method[method_name].shape}")
print(f" y_test dtype: {data['y_test'].dtype}, unique values: {np.unique(data['y_test'])}")


 KPCA_RBF test shape: (375, 100)
 KPCA_COSINE test shape: (375, 100)
 y_test dtype: object, unique values: [0 1]


In [18]:
def stage_4_final_xgboost_cv(X_train_scaled, y_train, X_val_scaled, y_val,
                             X_test_scaled, y_test, mlp_model,
                             top_k_indices, output_dir='deployment_grade',
                             n_splits=5, random_state=42):
    """
    Report 1: strict hold-out test performance (train+val -> test).
    Report 2: exploratory 5-fold stratified CV on merged data.
    Also trains one final model on all merged data for deployment.
    """
    os.makedirs(output_dir, exist_ok=True)

    y_train = y_train.astype(int)
    y_val = y_val.astype(int)
    y_test = y_test.astype(int)

    t0 = time.time()
    mlp_model.eval()
    with torch.no_grad():
        X_train_topk = mlp_model(torch.FloatTensor(X_train_scaled)).numpy()
        X_val_topk = mlp_model(torch.FloatTensor(X_val_scaled)).numpy()
        X_test_topk = mlp_model(torch.FloatTensor(X_test_scaled)).numpy()
    n_total = len(X_train_scaled) + len(X_val_scaled) + len(X_test_scaled)
    mlp_infer_per_sample_ms = (time.time() - t0) / max(n_total, 1) * 1000

    print('  Reproducibility settings: seed=42, deterministic torch=True, xgboost n_jobs=1')

    # Hold-out test
    X_trainval_topk = np.vstack([X_train_topk, X_val_topk])
    y_trainval = np.concatenate([y_train, y_val])

    n_pos_tv = int((y_trainval == 1).sum())
    n_neg_tv = int((y_trainval == 0).sum())
    scale_pos_weight_tv = float(n_neg_tv) / float(max(n_pos_tv, 1))
    n_samples_tv = len(y_trainval)
    w_neg_tv = n_samples_tv / (2.0 * max(n_neg_tv, 1))
    w_pos_tv = n_samples_tv / (2.0 * max(n_pos_tv, 1))
    sample_weight_tv = np.where(y_trainval == 1, w_pos_tv, w_neg_tv)

    xgb_holdout = xgb.XGBClassifier(
        n_estimators=350,
        max_depth=8,
        learning_rate=0.07,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=1,
        reg_lambda=5.0,
        gamma=0.0,
        scale_pos_weight=scale_pos_weight_tv,
        max_delta_step=3,
        random_state=random_state,
        n_jobs=1,
        eval_metric='logloss',
        verbose=0,
    )

    t0 = time.time()
    xgb_holdout.fit(X_trainval_topk, y_trainval, sample_weight=sample_weight_tv)
    holdout_train_time = time.time() - t0

    t0 = time.time()
    y_pred_holdout_test = xgb_holdout.predict(X_test_topk)
    y_pred_proba_holdout_test = xgb_holdout.predict_proba(X_test_topk)
    holdout_infer_per_sample_ms = (time.time() - t0) / max(len(X_test_topk), 1) * 1000

    holdout_accuracy = accuracy_score(y_test, y_pred_holdout_test)
    holdout_balanced_acc = balanced_accuracy_score(y_test, y_pred_holdout_test)
    holdout_f1_weighted = f1_score(y_test, y_pred_holdout_test, average='weighted')
    holdout_cm = confusion_matrix(y_test, y_pred_holdout_test)

    print('  Hold-out test summary (train+val -> test):')
    print(f'    Accuracy:          {holdout_accuracy:.4f}')
    print(f'    Balanced Accuracy: {holdout_balanced_acc:.4f}')
    print(f'    F1 (weighted):     {holdout_f1_weighted:.4f}')
    print(f'    Train time: {holdout_train_time:.2f}s | Infer time/sample: {holdout_infer_per_sample_ms:.3f}ms')

    holdout_labels = sorted(np.unique(y_test))
    fig, ax = plt.subplots(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(confusion_matrix=holdout_cm, display_labels=holdout_labels)
    disp.plot(ax=ax, colorbar=True, cmap='Blues')
    ax.set_title(
        f'Hold-out Test Confusion Matrix\n'
        f'Acc: {holdout_accuracy:.4f} | Bal Acc: {holdout_balanced_acc:.4f} | F1: {holdout_f1_weighted:.4f}',
        fontsize=12,
        pad=12,
    )
    plt.tight_layout()
    holdout_cm_path = os.path.join(output_dir, 'confusion_matrix_holdout_test.png')
    plt.savefig(holdout_cm_path, dpi=150, bbox_inches='tight')
    plt.close()

    # 5-fold CV on all data
    X_all_topk = np.vstack([X_train_topk, X_val_topk, X_test_topk])
    y_all = np.concatenate([y_train, y_val, y_test])

    print(f'  Merged dataset for CV: X={X_all_topk.shape}, y={y_all.shape}')
    print(f'  Running StratifiedKFold with n_splits={n_splits}, random_state={random_state}')

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    fold_metrics = []
    y_oof_pred = np.zeros_like(y_all)
    y_oof_proba = np.zeros(len(y_all), dtype=float)

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_all_topk, y_all), start=1):
        X_fold_train, X_fold_val = X_all_topk[train_idx], X_all_topk[val_idx]
        y_fold_train, y_fold_val = y_all[train_idx], y_all[val_idx]

        n_pos = int((y_fold_train == 1).sum())
        n_neg = int((y_fold_train == 0).sum())
        scale_pos_weight = float(n_neg) / float(max(n_pos, 1))
        n_samples = len(y_fold_train)
        w_neg = n_samples / (2.0 * max(n_neg, 1))
        w_pos = n_samples / (2.0 * max(n_pos, 1))
        sample_weight = np.where(y_fold_train == 1, w_pos, w_neg)

        xgb_fold = xgb.XGBClassifier(
            n_estimators=350,
            max_depth=8,
            learning_rate=0.07,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=1,
            reg_lambda=5.0,
            gamma=0.0,
            scale_pos_weight=scale_pos_weight,
            max_delta_step=3,
            random_state=random_state,
            n_jobs=1,
            eval_metric='logloss',
            verbose=0,
        )
        xgb_fold.fit(X_fold_train, y_fold_train, sample_weight=sample_weight)

        y_fold_pred = xgb_fold.predict(X_fold_val)
        y_fold_proba = xgb_fold.predict_proba(X_fold_val)[:, 1]
        fold_acc = accuracy_score(y_fold_val, y_fold_pred)
        fold_bal_acc = balanced_accuracy_score(y_fold_val, y_fold_pred)
        fold_f1 = f1_score(y_fold_val, y_fold_pred, average='weighted')

        fold_metrics.append({
            'fold': fold_idx,
            'accuracy': fold_acc,
            'balanced_accuracy': fold_bal_acc,
            'f1_weighted': fold_f1,
        })

        y_oof_pred[val_idx] = y_fold_pred
        y_oof_proba[val_idx] = y_fold_proba

        print(f'  Fold {fold_idx}: Acc={fold_acc:.4f}, BalAcc={fold_bal_acc:.4f}, F1={fold_f1:.4f}')

    oof_accuracy = accuracy_score(y_all, y_oof_pred)
    oof_balanced_acc = balanced_accuracy_score(y_all, y_oof_pred)
    oof_f1_weighted = f1_score(y_all, y_oof_pred, average='weighted')
    oof_cm = confusion_matrix(y_all, y_oof_pred)

    print(f'  CV Summary: Acc={oof_accuracy:.4f}, BalAcc={oof_balanced_acc:.4f}, F1={oof_f1_weighted:.4f}')

    oof_labels = sorted(np.unique(y_all))
    fig, ax = plt.subplots(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(confusion_matrix=oof_cm, display_labels=oof_labels)
    disp.plot(ax=ax, colorbar=True, cmap='Blues')
    ax.set_title(
        f'CV OOF Confusion Matrix\n'
        f'Acc: {oof_accuracy:.4f} | Bal Acc: {oof_balanced_acc:.4f} | F1: {oof_f1_weighted:.4f}',
        fontsize=12,
        pad=12,
    )
    plt.tight_layout()
    cm_path = os.path.join(output_dir, 'confusion_matrix_cv_oof.png')
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.close()

    # Train final model on all data
    n_pos_all = int((y_all == 1).sum())
    n_neg_all = int((y_all == 0).sum())
    scale_pos_weight_all = float(n_neg_all) / float(max(n_pos_all, 1))
    n_samples_all = len(y_all)
    w_neg_all = n_samples_all / (2.0 * max(n_neg_all, 1))
    w_pos_all = n_samples_all / (2.0 * max(n_pos_all, 1))
    sample_weight_all = np.where(y_all == 1, w_pos_all, w_neg_all)

    xgb_final = xgb.XGBClassifier(
        n_estimators=350,
        max_depth=8,
        learning_rate=0.07,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=1,
        reg_lambda=5.0,
        gamma=0.0,
        scale_pos_weight=scale_pos_weight_all,
        max_delta_step=3,
        random_state=random_state,
        n_jobs=1,
        eval_metric='logloss',
        verbose=0,
    )

    t0 = time.time()
    xgb_final.fit(X_all_topk, y_all, sample_weight=sample_weight_all)
    xgb_train_time = time.time() - t0

    t0 = time.time()
    _ = xgb_final.predict(X_all_topk)
    _ = xgb_final.predict_proba(X_all_topk)
    xgb_infer_per_sample_ms = (time.time() - t0) / max(len(X_all_topk), 1) * 1000

    explainer_final = shap.TreeExplainer(xgb_final)
    shap_values_final = explainer_final.shap_values(X_all_topk)
    if isinstance(shap_values_final, list):
        shap_values_final = shap_values_final[1]

    fig, ax = plt.subplots(figsize=(12, 8))
    shap.summary_plot(
        shap_values_final,
        X_all_topk,
        feature_names=[f'PredComp {top_k_indices[i]}' for i in range(len(top_k_indices))],
        plot_type='bar',
        show=False,
    )
    plt.tight_layout()
    shap_beeswarm_path = os.path.join(output_dir, 'shap_final_summary_bar_cv_full.png')
    plt.savefig(shap_beeswarm_path, dpi=150, bbox_inches='tight')
    plt.close()

    model_path = os.path.join(output_dir, 'xgboost_grade_predictor_cv.pkl')
    joblib.dump(xgb_final, model_path)

    result = {
        'xgb_final': xgb_final,
        'X_train_topk': X_train_topk,
        'X_val_topk': X_val_topk,
        'X_test_topk': X_test_topk,
        'X_all_topk': X_all_topk,
        'y_all': y_all,
        'y_pred_test': y_pred_holdout_test,
        'y_pred_proba_test': y_pred_proba_holdout_test,
        'holdout_metrics': {
            'accuracy': holdout_accuracy,
            'balanced_accuracy': holdout_balanced_acc,
            'f1_weighted': holdout_f1_weighted,
            'confusion_matrix': holdout_cm.tolist(),
            'cm_plot_path': holdout_cm_path,
        },
        'cv_metrics': {
            'accuracy': oof_accuracy,
            'balanced_accuracy': oof_balanced_acc,
            'f1_weighted': oof_f1_weighted,
            'confusion_matrix': oof_cm.tolist(),
            'cm_plot_path': cm_path,
            'fold_metrics': fold_metrics,
        },
        'y_oof_pred': y_oof_pred,
        'y_oof_proba': y_oof_proba,
        'model_path': model_path,
        'shap_plot_path': shap_beeswarm_path,
        'mlp_infer_per_sample_ms': mlp_infer_per_sample_ms,
        'xgb_infer_per_sample_ms': xgb_infer_per_sample_ms,
        'xgb_train_time_s': xgb_train_time,
        'settings': {
            'random_state': int(random_state),
            'torch_deterministic': True,
            'xgboost_n_jobs': 1,
            'cv_n_splits': int(n_splits),
        },
    }

    return result


xgb_final_results = {}
for method_name, cr in component_results.items():
    print(f"\n=== Stage 4 (KMeans ranking) for {cr['component_name']} ({method_name}) ===")
    xgb_final_results[method_name] = stage_4_final_xgboost_cv(
        data['X_train'], data['y_train'],
        data['X_val'], data['y_val'],
        data['X_test'], data['y_test'],
        mlp_results[method_name]['model'],
        shap_results[method_name]['top_k_indices'],
        output_dir=f"deployment_grade/{cr['component_name']}",
        n_splits=5,
        random_state=42,
    )


xgb_final_result = xgb_final_results[ACTIVE_COMPONENT_METHOD]



=== Stage 4 (KMeans ranking) for KPCA_RBF (kpca_rbf) ===
  Reproducibility settings: seed=42, deterministic torch=True, xgboost n_jobs=1
  Hold-out test summary (train+val -> test):
    Accuracy:          0.8827
    Balanced Accuracy: 0.8691
    F1 (weighted):     0.8940
    Train time: 0.88s | Infer time/sample: 0.017ms
  Merged dataset for CV: X=(2494, 50), y=(2494,)
  Running StratifiedKFold with n_splits=5, random_state=42
  Fold 1: Acc=0.8637, BalAcc=0.8461, F1=0.8784
  Fold 2: Acc=0.8818, BalAcc=0.8356, F1=0.8914
  Fold 3: Acc=0.8697, BalAcc=0.8032, F1=0.8797
  Fold 4: Acc=0.8657, BalAcc=0.7738, F1=0.8743
  Fold 5: Acc=0.8936, BalAcc=0.8908, F1=0.9038
  CV Summary: Acc=0.8749, BalAcc=0.8296, F1=0.8857

=== Stage 4 (KMeans ranking) for KPCA_COSINE (kpca_cosine) ===
  Reproducibility settings: seed=42, deterministic torch=True, xgboost n_jobs=1
  Hold-out test summary (train+val -> test):
    Accuracy:          0.8720
    Balanced Accuracy: 0.8448
    F1 (weighted):     0.8844
   

In [75]:

def stage_5_gene_mapping(X_train_raw, X_train_components, top_k_indices,
                         gene_names, component_name, output_dir='deployment_grade'):
    """
    For each top-K component, train LinearRegression: raw genes -> component value.
    Extract top-10 genes per component by absolute coefficient.
    """
    
    print(f"  Training LinearRegression for each of {len(top_k_indices)} components...")
    print(f"  X_train_raw shape: {X_train_raw.shape}, gene_names: {len(gene_names)}")
    
    component_gene_mapping = {}
    
    if len(gene_names) != X_train_raw.shape[1]:
        raise ValueError(
            f"gene_names length ({len(gene_names)}) must match number of features ({X_train_raw.shape[1]})"
        )
    
    for i, component_idx in enumerate(top_k_indices):
        y_component = X_train_components[:, component_idx]
        lr = LinearRegression()
        lr.fit(X_train_raw, y_component)
        
        abs_coefs = np.abs(lr.coef_)
        top_10_indices = np.argsort(abs_coefs)[-10:][::-1]
        
        top_genes = [
            {
                'gene_name': str(gene_names[int(gene_idx)]),
                'coefficient': float(lr.coef_[gene_idx]),
                'abs_coefficient': float(abs_coefs[gene_idx])
            }
            for gene_idx in top_10_indices
        ]
        
        component_gene_mapping[f'component_{component_idx}'] = {
            'intercept': float(lr.intercept_),
            'top_10_genes': top_genes
        }
        
        if (i + 1) % 5 == 0:
            print(f"    Processed {i+1}/{len(top_k_indices)} components...")
    
    os.makedirs(output_dir, exist_ok=True)
    safe_name = component_name.lower().replace(' ', '_').replace('-', '_')
    mapping_path = os.path.join(output_dir, f'component_gene_mapping_{safe_name}.json')
    with open(mapping_path, 'w') as f:
        json.dump(component_gene_mapping, f, indent=2)
    
    print(f" STAGE 5: Gene Mapping Complete ({component_name})")
    print(f"  Mapping saved to: {mapping_path}")
    
    return component_gene_mapping

gene_names = clean_rnaseq_df['gene_id'].tolist()
X_train_raw = data['scaler'].inverse_transform(data['X_train'])

gene_mappings = {}
for method_name, cr in component_results.items():
    print(f"\n=== Stage 5 for {cr['component_name']} ({method_name}) ===")
    gene_mappings[method_name] = stage_5_gene_mapping(
        X_train_raw,
        cr['X_train_components'],
        shap_results[method_name]['top_k_indices'],
        gene_names,
        component_name=cr['component_name'],
        output_dir='deployment_grade'
    )

# Backward-compatible alias for active method
gene_mapping = gene_mappings[ACTIVE_COMPONENT_METHOD]

print("Example mapping for first component (active method):")
first_component = f"component_{shap_results[ACTIVE_COMPONENT_METHOD]['top_k_indices'][0]}"
print(json.dumps({first_component: gene_mapping[first_component]}, indent=2))



=== Stage 5 for KPCA_RBF (kpca_rbf) ===
  Training LinearRegression for each of 50 components...
  X_train_raw shape: (1745, 19310), gene_names: 19310
    Processed 5/50 components...
    Processed 10/50 components...
    Processed 15/50 components...
    Processed 20/50 components...
    Processed 25/50 components...
    Processed 30/50 components...
    Processed 35/50 components...
    Processed 40/50 components...
    Processed 45/50 components...
    Processed 50/50 components...
 STAGE 5: Gene Mapping Complete (KPCA_RBF)
  Mapping saved to: deployment_grade/component_gene_mapping_kpca_rbf.json

=== Stage 5 for KPCA_COSINE (kpca_cosine) ===
  Training LinearRegression for each of 50 components...
  X_train_raw shape: (1745, 19310), gene_names: 19310
    Processed 5/50 components...
    Processed 10/50 components...
    Processed 15/50 components...
    Processed 20/50 components...
    Processed 25/50 components...
    Processed 30/50 components...
    Processed 35/50 components.